In [ ]:
import os
from huggingface_hub import snapshot_download

model_name = "unsloth/Qwen3-4B-bnb-4bit"  # Correct unsloth 4-bit model repo
save_dir = os.path.join(os.getcwd(), "models", "Qwen3-4B-unsloth-4bit")

os.makedirs(save_dir, exist_ok=True)
print(f"Model will be downloaded: {save_dir}")

if os.path.exists(save_dir) and os.listdir(save_dir):
    print("Model already exists, skipping installation.")
else:
    snapshot_download(
        repo_id=model_name,
        local_dir=save_dir,
        local_dir_use_symlinks=False
    )
    print("\nDownloaded successfully!")

print(f"Files: {os.listdir(save_dir)}")

Model will be downloaded: /home/yesoytur/APilus/models/Qwen3-4B-unsloth-4bit
Model already exists, skipping installation.
Files: ['.cache', 'chat_template.jinja', 'generation_config.json', 'config.json', '.gitattributes', 'vocab.json', 'added_tokens.json', 'special_tokens_map.json', 'model.safetensors', 'tokenizer_config.json', 'merges.txt', 'tokenizer.json', 'README.md']


In [4]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import os

model_path = os.path.join(os.getcwd(), "models", "Qwen3-4B-unsloth-4bit")

tokenizer = AutoTokenizer.from_pretrained(model_path)

# Correct way to load 4-bit quantized models
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",  # Computation dtype during inference
    bnb_4bit_use_double_quant=True,    # Extra memory savings with nested quantization
    bnb_4bit_quant_type="nf4"         # NF4 is the standard for unsloth models
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto"
)

def ask(prompt, enable_thinking=False, max_new_tokens=512):
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    generated_ids = model.generate(**model_inputs, max_new_tokens=max_new_tokens)
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):]
    return tokenizer.decode(output_ids, skip_special_tokens=True).strip()

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [3]:
import time

# Estimate the maximum number of tokens the model will use for a given prompt
# Also benchmarks real generation speed (tokens per second)

def estimate_max_tokens(prompt, enable_thinking=False, max_new_tokens=512):
    # Tokenize the prompt using the chat template (same as ask())
    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking
    )

    # Count input (prompt) tokens
    input_ids = tokenizer([text], return_tensors="pt").to(model.device)
    input_token_count = input_ids.input_ids.shape[1]

    # Get the model's absolute maximum context length from its config
    model_max_context = model.config.max_position_embeddings

    # The model can generate at most max_new_tokens, but also cannot
    # exceed the remaining space in the context window
    remaining_context = model_max_context - input_token_count
    effective_max_output = min(max_new_tokens, remaining_context)

    # Total tokens the model will work with at peak (input + max possible output)
    total_max_tokens = input_token_count + effective_max_output

    # Run actual generation to measure real tokens per second
    start_time = time.perf_counter()
    generated_ids = model.generate(**input_ids, max_new_tokens=max_new_tokens)
    elapsed = time.perf_counter() - start_time

    # Count only the newly generated tokens (excluding the prompt)
    output_ids = generated_ids[0][input_token_count:]
    actual_output_tokens = len(output_ids)
    tokens_per_second = actual_output_tokens / elapsed

    print(f"\n{'─'*45}")
    print(f"  🧠 Model context limit  : {model_max_context:>7} tokens")
    print(f"  📥 Input tokens         : {input_token_count:>7} tokens")
    print(f"  📤 Max new tokens (arg) : {max_new_tokens:>7} tokens")
    print(f"  📭 Remaining context    : {remaining_context:>7} tokens")
    print(f"  ✅ Effective max output : {effective_max_output:>7} tokens")
    print(f"  🔢 Total max tokens     : {total_max_tokens:>7} tokens")
    print(f"  📝 Actual output tokens : {actual_output_tokens:>7} tokens")
    print(f"  ⏱  Generation time      : {elapsed:>7.2f} s")
    print(f"  ⚡ Tokens per second    : {tokens_per_second:>7.1f} tok/s")
    print(f"{'─'*45}\n")

    return {
        "model_max_context": model_max_context,
        "input_tokens": input_token_count,
        "max_new_tokens_arg": max_new_tokens,
        "remaining_context": remaining_context,
        "effective_max_output": effective_max_output,
        "total_max_tokens": total_max_tokens,
        "actual_output_tokens": actual_output_tokens,
        "elapsed_sec": round(elapsed, 3),
        "tokens_per_second": round(tokens_per_second, 2),
    }

# Example usage
estimate_max_tokens("Explain me raycasting")

Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



─────────────────────────────────────────────
  🧠 Model context limit  :   40960 tokens
  📥 Input tokens         :      17 tokens
  📤 Max new tokens (arg) :     512 tokens
  📭 Remaining context    :   40943 tokens
  ✅ Effective max output :     512 tokens
  🔢 Total max tokens     :     529 tokens
  📝 Actual output tokens :     512 tokens
  ⏱  Generation time      :   12.55 s
  ⚡ Tokens per second    :    40.8 tok/s
─────────────────────────────────────────────



{'model_max_context': 40960,
 'input_tokens': 17,
 'max_new_tokens_arg': 512,
 'remaining_context': 40943,
 'effective_max_output': 512,
 'total_max_tokens': 529,
 'actual_output_tokens': 512,
 'elapsed_sec': 12.553,
 'tokens_per_second': 40.79}

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig, GenerationConfig
import os
import time

model_path = os.path.join(os.getcwd(), "models", "Qwen3-4B-unsloth-4bit")

tokenizer = AutoTokenizer.from_pretrained(model_path)

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype="float16",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_quant_type="nf4"
)

model = AutoModelForCausalLM.from_pretrained(
    model_path,
    quantization_config=bnb_config,
    device_map="auto"
)

def ask(
    prompt,
    enable_thinking=False,
    max_new_tokens=512,

    # --- Sampling ---
    temperature=0.7,
    top_p=0.9,
    top_k=50,
    repetition_penalty=1.1,

    # --- Determinism ---
    do_sample=True,
    seed=None,

    # --- Output ---
    verbose=True,
):
    if seed is not None:
        import torch
        torch.manual_seed(seed)

    messages = [{"role": "user", "content": prompt}]
    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
        enable_thinking=enable_thinking
    )

    model_inputs = tokenizer([text], return_tensors="pt").to(model.device)
    input_token_count = model_inputs.input_ids.shape[1]

    gen_config = GenerationConfig(
        max_new_tokens=max_new_tokens,
        do_sample=do_sample,
        temperature=temperature if do_sample else None,
        top_p=top_p if do_sample else None,
        top_k=top_k if do_sample else None,
        repetition_penalty=repetition_penalty,
        pad_token_id=tokenizer.eos_token_id,
    )

    start_time = time.perf_counter()
    generated_ids = model.generate(**model_inputs, generation_config=gen_config)
    elapsed = time.perf_counter() - start_time

    output_ids = generated_ids[0][input_token_count:]
    output_token_count = len(output_ids)
    tokens_per_second = output_token_count / elapsed

    response = tokenizer.decode(output_ids, skip_special_tokens=True).strip()

    if verbose:
        print(f"\n{'─'*48}")
        print(f"  🌡  Temperature         : {temperature}")
        print(f"  🎯 Top-p               : {top_p}")
        print(f"  🔝 Top-k               : {top_k}")
        print(f"  🔁 Repetition penalty  : {repetition_penalty}")
        print(f"  🎲 Sampling            : {do_sample}  |  Seed: {seed}")
        print(f"  {'─'*42}")
        print(f"  📥 Input tokens        : {input_token_count:>6} tokens")
        print(f"  📤 Output tokens       : {output_token_count:>6} tokens")
        print(f"  ⏱  Generation time     : {elapsed:>6.2f} s")
        print(f"  ⚡ Tokens per second   : {tokens_per_second:>6.1f} tok/s")
        print(f"{'─'*48}\n")

    return response

# Example usage
print(ask("Tell me a joke"))

print(ask("compare TCP and UDP", temperature=0.8))

/home/yesoytur/APilus/.venv/lib/python3.11/site-packages/transformers/quantizers/auto.py:262: UserWarning: You passed `quantization_config` or equivalent parameters to `from_pretrained` but the model you're loading already has a `quantization_config` attribute. The `quantization_config` from the model will be used.
  warnings.warn(warning_msg)


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=512) and `max_length`(=40960) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)



────────────────────────────────────────────────
  🌡  Temperature         : 0.7
  🎯 Top-p               : 0.9
  🔝 Top-k               : 50
  🔁 Repetition penalty  : 1.1
  🎲 Sampling            : True  |  Seed: None
  ──────────────────────────────────────────
  📥 Input tokens        :     16 tokens
  📤 Output tokens       :     20 tokens
  ⏱  Generation time     :   0.53 s
  ⚡ Tokens per second   :   38.0 tok/s
────────────────────────────────────────────────

Why don't skeletons fight each other?  
Because they don't have the guts! 😄

────────────────────────────────────────────────
  🌡  Temperature         : 0.8
  🎯 Top-p               : 0.9
  🔝 Top-k               : 50
  🔁 Repetition penalty  : 1.1
  🎲 Sampling            : True  |  Seed: None
  ──────────────────────────────────────────
  📥 Input tokens        :     16 tokens
  📤 Output tokens       :    512 tokens
  ⏱  Generation time     :  12.29 s
  ⚡ Tokens per second   :   41.7 tok/s
──────────────────────────────────────────

In [5]:
print(ask("What is multiplexing?"))

KeyboardInterrupt: 